# Spike Reference Sensitivity

Fit models using each homolog as reference and compare inferred parameters.

In [ ]:
config_path = "config/config.yaml"
profile = None

In [ ]:
import warnings
import time
import pickle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
import multidms

from notebooks._common import load_config, combine_replicate_muts, build_fit_params


In [ ]:
cfg = load_config(config_path, profile)
spike = cfg["spike"]
seed = cfg["seed"]
fit = spike["fitting"]

output_dir = spike["output_dir"]
condition_colors = spike["condition_colors"]
chosen_lasso_strength = spike["chosen_lasso_strength"]
times_seen_threshold = spike["times_seen_threshold"]
gpu_ids = cfg["gpu_ids"]
n_processes = cfg["n_processes"]

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
%matplotlib inline
plt.rcParams.update({"legend.frameon": False, "font.size": 11})

In [ ]:
func_score_df = pd.read_csv(f"{output_dir}/training_functional_scores.csv").fillna({"aa_substitutions": ""})

subsample_frac = spike.get("subsample_frac", None)
if subsample_frac is not None:
    func_score_df = (
        pd.concat([
        g.sample(frac=subsample_frac, random_state=seed)
        for _, g in func_score_df.groupby(["condition", "replicate"])
    ]).reset_index(drop=True)
    )
print(f"Loaded {len(func_score_df)} variants")

## Fit models with each reference

In [ ]:
cc = condition_colors
variable_reference_datasets = []

for reference in ["Delta", "Omicron_BA1", "Omicron_BA2"]:
    for replicate, rep_fsdf in func_score_df.groupby("replicate"):
        df_agg = (
            rep_fsdf.groupby(["condition", "aa_substitutions"], dropna=False)
            .agg({"func_score": "mean"})
            .reset_index()
        )
        df_agg["aa_substitutions"] = df_agg["aa_substitutions"].fillna("")
        data = multidms.Data(
            df_agg,
            alphabet=multidms.AAS_WITHSTOP_WITHGAP,
            reference=reference,
            assert_site_integrity=False,
            verbose=False,
            name=f"{replicate}-{reference}",
        )
        data.condition_colors = cc
        variable_reference_datasets.append(data)

variable_reference_fit_params = build_fit_params(fit, variable_reference_datasets)
variable_reference_fit_params["fusionreg"] = [chosen_lasso_strength]

_, _, variable_reference_models = multidms.fit_models(
    variable_reference_fit_params, gpu_ids=gpu_ids, n_processes=n_processes
)
for col in variable_reference_models.columns:
    if variable_reference_models[col].apply(lambda x: isinstance(x, dict)).any():
        variable_reference_models[col] = variable_reference_models[col].apply(str)

variable_reference_models = variable_reference_models.assign(
    reference=lambda x: x.dataset_name.str.split("-").str[-1],
    replicate=lambda x: x.dataset_name.str.split("-").str[0],
)
print(f"Fit {len(variable_reference_models)} reference-sensitivity models")

## Compute relative parameters

In [ ]:
homologs = ["Delta", "Omicron_BA1", "Omicron_BA2"]
relative_params = pd.DataFrame()

for reference, replicate_models in variable_reference_models.groupby("reference"):
    mut_df = combine_replicate_muts(
        {f"rep_{row.replicate}": row["model"] for idx, row in replicate_models.iterrows()},
        times_seen_threshold=times_seen_threshold,
    )
    mut_df = mut_df.copy()[[c for c in mut_df.columns if "avg" in c]]
    for homolog in homologs:
        if homolog == reference:
            mut_df[f"beta_{homolog}"] = mut_df[f"avg_beta_{reference}"]
        else:
            mut_df[f"beta_{homolog}"] = mut_df[f"avg_beta_{reference}"] + mut_df[f"avg_shift_{homolog}"]
    for homolog in homologs:
        mut_df[f"shift_{homolog}"] = mut_df[f"beta_{homolog}"] - mut_df["beta_Omicron_BA1"]
    mut_df.drop([c for c in mut_df.columns if "avg" in c], axis=1, inplace=True)
    mut_df = mut_df.assign(reference=reference)
    relative_params = pd.concat([relative_params, mut_df])

relative_params.drop(["beta_Delta", "beta_Omicron_BA2", "shift_Omicron_BA1"], axis=1, inplace=True, errors="ignore")
relative_params.reference.replace({"Omicron_BA2": "BA2", "Omicron_BA1": "BA1"}, inplace=True)
print(f"Relative params: {len(relative_params)} rows")

## Reference comparison scatter

In [ ]:
saveas = "reference_model_comparison_params_scatter"
parameters = ["beta_Omicron_BA1", "shift_Delta", "shift_Omicron_BA2"]
references = relative_params.reference.unique()
ref_pairs = [(r1, r2) for i, r1 in enumerate(references) for r2 in references[i + 1:]]

fig, axes = plt.subplots(len(ref_pairs), len(parameters), figsize=[9, 3 * len(ref_pairs)], squeeze=False)

for row, (ref1, ref2) in enumerate(ref_pairs):
    df1 = relative_params.query(f"reference == '{ref1}'")
    df2 = relative_params.query(f"reference == '{ref2}'")
    common = df1.index.intersection(df2.index)

    for col, param in enumerate(parameters):
        iter_ax = axes[row, col]
        x = df1.loc[common, param]
        y = df2.loc[common, param]
        iter_ax.scatter(x, y, alpha=0.3, s=10, c="0.25")
        lim = [min(x.min(), y.min()) * 1.1, max(x.max(), y.max()) * 1.1]
        iter_ax.plot(lim, lim, "--", c="royalblue", lw=1)
        corr = pearsonr(x.dropna(), y.dropna())[0] ** 2
        iter_ax.annotate(f"$R^2 = {corr:.2f}$", (0.05, 0.9), xycoords="axes fraction", fontsize=10)
        if row == 0:
            iter_ax.set_title(param.replace("beta_", "β ").replace("shift_", "Δ "))
        if col == 0:
            iter_ax.set_ylabel(f"ref={ref2}")
        iter_ax.set_xlabel(f"ref={ref1}" if row == len(ref_pairs) - 1 else "")
        sns.despine(ax=iter_ax)

plt.tight_layout()
fig.savefig(f"{output_dir}/{saveas}.pdf", bbox_inches="tight")
plt.show()